# SER to FITS converter
Written by Kiyoaki Okudaira<br>
*Kyushu University Hanada Lab / University of Washington / IAU CPS SatHub<br>
(okudaira.kiyoaki.528@s.kyushu-u.ac.jp or kiyoaki@uw.edu)<br>
<br>
Conver SER file to FITS file (single frame).<br>
<br>
**History**<br>
coding 2026-07-08 : 1st coding<br>
<br>
(c) 2026 Kiyoaki Okudaira - Kyushu University Hanada Lab (SSDL) / University of Washington / IAU CPS SatHub

### Settings

In [ ]:
from pathlib import Path

PATH_ser = Path(
    "/Volumes/SSD-PEU4A/Images/2026-04-05/EKRAN 12/19_45_10.ser"
)

# CameraSettings file.
# Set to None if no CameraSettings file is available.
PATH_camera_settings = Path(
    "/Volumes/SSD-PEU4A/Images/2026-04-05/EKRAN 12/19_45_10.CameraSettings.txt"
)

# Zero-based frame index
frame_index = 946

PATH_output_directory = Path(
    "/Users/kiyoaki/VScode/satphotometry_photometry/output/ser2fits"
)

### Run

In [2]:
from datetime import datetime, timezone
import re

from astropy.io import fits
from satphotometry import serparser


frame_index = frame_index - 1
# ============================================================
# Utility functions
# ============================================================

def parse_camera_settings(path):
    """
    Read a SharpCap CameraSettings text file.

    Parameters
    ----------
    path : str or pathlib.Path
        Path to the CameraSettings file.

    Returns
    -------
    dict
        Dictionary containing key-value pairs.
    """
    path = Path(path)

    settings = {}
    current_section = None

    with path.open("r", encoding="utf-8-sig") as file:
        for raw_line in file:
            line = raw_line.strip()

            # Ignore blank lines and comment lines
            if not line or line.startswith("#"):
                continue

            # Section name, such as [QHY461M]
            if line.startswith("[") and line.endswith("]"):
                current_section = line[1:-1].strip()
                settings["Camera Model"] = current_section
                continue

            if "=" not in line:
                continue

            key, value = line.split("=", 1)

            settings[key.strip()] = value.strip()

    return settings


def parse_numeric_value(value):
    """
    Extract the first numerical value from a string.

    Examples
    --------
    '125.0000ms' -> 125.0
    '3.4492fps'  -> 3.4492
    '-9.9'       -> -9.9
    """
    if value is None:
        return None

    match = re.search(
        r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?",
        str(value),
    )

    if match is None:
        return None

    return float(match.group())


def parse_iso_datetime(value):
    """
    Convert a SharpCap ISO-format datetime string to a Python datetime.

    SharpCap may use seven fractional-second digits, whereas Python supports
    up to six. The final fractional digit is therefore removed if necessary.
    """
    if not value:
        return None

    value = value.strip()

    # Replace UTC suffix
    if value.endswith("Z"):
        value = value[:-1] + "+00:00"

    # Limit fractional seconds to six digits
    value = re.sub(
        r"(\.\d{6})\d+(?=[+-]\d{2}:\d{2}$)",
        r"\1",
        value,
    )

    try:
        return datetime.fromisoformat(value)
    except ValueError:
        return None


def parse_binning(value):
    """
    Convert a binning string such as '2x2' into two integers.
    """
    if not value:
        return None, None

    match = re.fullmatch(
        r"\s*(\d+)\s*[xX×]\s*(\d+)\s*",
        value,
    )

    if match is None:
        return None, None

    return int(match.group(1)), int(match.group(2))


def parse_capture_area(value):
    """
    Convert a capture-area string such as '5880x4422' into width and height.
    """
    if not value:
        return None, None

    match = re.fullmatch(
        r"\s*(\d+)\s*[xX×]\s*(\d+)\s*",
        value,
    )

    if match is None:
        return None, None

    return int(match.group(1)), int(match.group(2))


def add_camera_settings_to_header(header, settings, frame_index=None):
    """
    Add selected SharpCap CameraSettings metadata to a FITS header.

    FITS keyword names are kept at eight characters or fewer where practical.
    Longer or less-standard metadata is stored using HIERARCH keywords.
    """

    # --------------------------------------------------------
    # Camera and acquisition information
    # --------------------------------------------------------

    if settings.get("Camera Model"):
        header["INSTRUME"] = (
            settings["Camera Model"],
            "Camera model",
        )

    if settings.get("CameraSerialNumber"):
        header["HIERARCH CAMERA SERIAL"] = (
            settings["CameraSerialNumber"],
            "Camera serial number",
        )

    if settings.get("FrameType"):
        header["IMAGETYP"] = (
            settings["FrameType"],
            "Image frame type",
        )

    if settings.get("Colour Space"):
        header["COLSPACE"] = (
            settings["Colour Space"],
            "Camera colour space",
        )

    if settings.get("Read Mode"):
        header["READMODE"] = (
            settings["Read Mode"],
            "Camera readout mode",
        )

    if settings.get("Output Format"):
        header["HIERARCH OUTPUT FORMAT"] = (
            settings["Output Format"],
            "Original capture output format",
        )

    # --------------------------------------------------------
    # Exposure and detector settings
    # --------------------------------------------------------

    exposure_ms = parse_numeric_value(
        settings.get("Exposure")
    )

    if exposure_ms is not None:
        header["EXPTIME"] = (
            exposure_ms / 1000.0,
            "Exposure time [s]",
        )

    gain = parse_numeric_value(
        settings.get("Gain")
    )

    if gain is not None:
        header["GAIN"] = (
            gain,
            "Camera gain setting",
        )

    offset = parse_numeric_value(
        settings.get("Offset")
    )

    if offset is not None:
        header["OFFSET"] = (
            offset,
            "Camera offset setting",
        )

    usb_traffic = parse_numeric_value(
        settings.get("USB Traffic")
    )

    if usb_traffic is not None:
        header["USBTRFFC"] = (
            usb_traffic,
            "USB traffic setting",
        )

    temperature = parse_numeric_value(
        settings.get("Temperature")
    )

    if temperature is not None:
        header["CCD-TEMP"] = (
            temperature,
            "Sensor temperature [deg C]",
        )

    target_temperature = parse_numeric_value(
        settings.get("Target Temperature")
    )

    if target_temperature is not None:
        header["SET-TEMP"] = (
            target_temperature,
            "Target sensor temperature [deg C]",
        )

    humidity = parse_numeric_value(
        settings.get("Sensor Relative Humidity")
    )

    if humidity is not None:
        header["HIERARCH SENSOR HUMIDITY"] = (
            humidity,
            "Sensor relative humidity [%]",
        )

    if settings.get("Cooler Power"):
        header["HIERARCH COOLER POWER"] = (
            settings["Cooler Power"],
            "Camera cooler power setting",
        )

    # --------------------------------------------------------
    # Image geometry
    # --------------------------------------------------------

    bin_x, bin_y = parse_binning(
        settings.get("Binning")
    )

    if bin_x is not None:
        header["XBINNING"] = (
            bin_x,
            "Horizontal binning",
        )

    if bin_y is not None:
        header["YBINNING"] = (
            bin_y,
            "Vertical binning",
        )

    capture_width, capture_height = parse_capture_area(
        settings.get("Capture Area")
    )

    if capture_width is not None:
        header["HIERARCH CAPTURE WIDTH"] = (
            capture_width,
            "Capture-area width [pixel]",
        )

    if capture_height is not None:
        header["HIERARCH CAPTURE HEIGHT"] = (
            capture_height,
            "Capture-area height [pixel]",
        )

    pan = parse_numeric_value(
        settings.get("Pan")
    )

    if pan is not None:
        header["XORGSUBF"] = (
            int(pan),
            "Subframe horizontal origin [pixel]",
        )

    tilt = parse_numeric_value(
        settings.get("Tilt")
    )

    if tilt is not None:
        header["YORGSUBF"] = (
            int(tilt),
            "Subframe vertical origin [pixel]",
        )

    # --------------------------------------------------------
    # Capture timing
    # --------------------------------------------------------

    start_capture = parse_iso_datetime(
        settings.get("StartCapture")
    )

    mid_capture = parse_iso_datetime(
        settings.get("MidCapture")
    )

    end_capture = parse_iso_datetime(
        settings.get("EndCapture")
    )

    if start_capture is not None:
        header["DATE-OBS"] = (
            start_capture.isoformat(),
            "Capture start time [UTC]",
        )

    if mid_capture is not None:
        header["DATE-MID"] = (
            mid_capture.isoformat(),
            "Capture midpoint time [UTC]",
        )

    if end_capture is not None:
        header["DATE-END"] = (
            end_capture.isoformat(),
            "Capture end time [UTC]",
        )

    jd_start = parse_numeric_value(
        settings.get("JDStartCapture")
    )

    if jd_start is not None:
        header["JD-OBS"] = (
            jd_start,
            "Julian Date at capture start",
        )

    jd_mid = parse_numeric_value(
        settings.get("JDMidCapture")
    )

    if jd_mid is not None:
        header["JD-MID"] = (
            jd_mid,
            "Julian Date at capture midpoint",
        )

    jd_end = parse_numeric_value(
        settings.get("JDEndCapture")
    )

    if jd_end is not None:
        header["JD-END"] = (
            jd_end,
            "Julian Date at capture end",
        )

    duration = parse_numeric_value(
        settings.get("Duration")
    )

    if duration is not None:
        header["DURATION"] = (
            duration,
            "Total capture duration [s]",
        )

    frame_count = parse_numeric_value(
        settings.get("FrameCount")
    )

    if frame_count is not None:
        header["NFRAMES"] = (
            int(frame_count),
            "Number of frames in capture",
        )

    frame_rate = parse_numeric_value(
        settings.get("ActualFrameRate")
    )

    if frame_rate is not None:
        header["FRAMERAT"] = (
            frame_rate,
            "Actual frame rate [frame/s]",
        )

    time_zone = parse_numeric_value(
        settings.get("TimeZone")
    )

    if time_zone is not None:
        header["HIERARCH TIME ZONE"] = (
            time_zone,
            "Local time-zone offset [hour]",
        )

    # --------------------------------------------------------
    # Software information
    # --------------------------------------------------------

    if settings.get("SharpCapVersion"):
        header["SWCREATE"] = (
            f"SharpCap {settings['SharpCapVersion']}",
            "Capture software",
        )

    if settings.get("TimeStamp"):
        header["HIERARCH SETTINGS TIMESTAMP"] = (
            settings["TimeStamp"],
            "CameraSettings creation time",
        )

    # --------------------------------------------------------
    # Plate-solve information
    # --------------------------------------------------------

    if settings.get("LastPlateSolveData"):
        header["HIERARCH LAST PLATE SOLVE"] = (
            settings["LastPlateSolveData"],
            "Last SharpCap plate-solve result",
        )

    # --------------------------------------------------------
    # Frame-specific information
    # --------------------------------------------------------

    if frame_index is not None:
        header["FRAMEIDX"] = (
            int(frame_index),
            "Zero-based SER frame index",
        )

    # --------------------------------------------------------
    # Store all remaining settings as HIERARCH keywords
    # --------------------------------------------------------

    already_handled = {
        "Camera Model",
        "CameraSerialNumber",
        "FrameType",
        "Colour Space",
        "Read Mode",
        "Output Format",
        "Exposure",
        "Gain",
        "Offset",
        "USB Traffic",
        "Temperature",
        "Target Temperature",
        "Sensor Relative Humidity",
        "Cooler Power",
        "Binning",
        "Capture Area",
        "Pan",
        "Tilt",
        "StartCapture",
        "MidCapture",
        "EndCapture",
        "JDStartCapture",
        "JDMidCapture",
        "JDEndCapture",
        "Duration",
        "FrameCount",
        "ActualFrameRate",
        "TimeZone",
        "SharpCapVersion",
        "TimeStamp",
        "LastPlateSolveData",
    }

    for key, value in settings.items():
        if key in already_handled:
            continue

        if value == "":
            continue

        # HIERARCH permits descriptive FITS keyword names longer than
        # the standard eight-character FITS limit.
        fits_key = f"HIERARCH SHARPCAP {key.upper()}"

        try:
            header[fits_key] = (
                value,
                "SharpCap CameraSettings",
            )
        except Exception:
            # Skip values that cannot safely be stored in the FITS header
            pass

    return header


# ============================================================
# Open the SER file
# ============================================================

if not PATH_ser.is_file():
    raise FileNotFoundError(
        f"SER file was not found: {PATH_ser}"
    )

ser_file = serparser.Serfile(
    str(PATH_ser)
)

frame_count = int(
    ser_file.getLength()
)

if not 0 <= frame_index < frame_count:
    raise IndexError(
        f"frame_index must be between 0 and {frame_count - 1}. "
        f"Received: {frame_index}"
    )


# ============================================================
# Read the specified frame
# ============================================================

frame_data = ser_file.readFrameAtPos(
    frame_index
)

if not hasattr(frame_data, "shape"):
    raise RuntimeError(
        f"Failed to read frame {frame_index}."
    )


# ============================================================
# Create the FITS header
# ============================================================

fits_header = ser_file.createFitsHeader()

fits_header["FILENAME"] = (
    PATH_ser.name,
    "Original SER filename",
)

fits_header["FRAMEIDX"] = (
    frame_index,
    "Zero-based SER frame index",
)


# ============================================================
# Read CameraSettings metadata when available
# ============================================================

camera_settings_loaded = False

if (
    PATH_camera_settings is not None
    and Path(PATH_camera_settings).is_file()
):
    camera_settings = parse_camera_settings(
        PATH_camera_settings
    )

    fits_header = add_camera_settings_to_header(
        header=fits_header,
        settings=camera_settings,
        frame_index=frame_index,
    )

    camera_settings_loaded = True

    print(
        f"CameraSettings loaded: {PATH_camera_settings}"
    )

else:
    print(
        "CameraSettings file was not found. "
        "The FITS file will be written without CameraSettings metadata."
    )


# ============================================================
# Write the FITS file
# ============================================================

PATH_output_directory.mkdir(
    parents=True,
    exist_ok=True,
)

PATH_output_fits = (
    PATH_output_directory
    / f"{PATH_ser.stem}_frame_{(frame_index+1):06d}.fits"
)

primary_hdu = fits.PrimaryHDU(
    data=frame_data,
    header=fits_header,
)

primary_hdu.writeto(
    PATH_output_fits,
    overwrite=True,
    output_verify="fix",
)


# ============================================================
# Results
# ============================================================

print()
print("FITS export completed.")
print(f"Input SER       : {PATH_ser}")
print(f"Frame index     : {frame_index}")
print(f"Frame count     : {frame_count}")
print(
    f"Image size      : "
    f"{ser_file.getWidth()} × {ser_file.getHeight()} pixels"
)
print(f"Camera metadata : {camera_settings_loaded}")
print(f"Output FITS     : {PATH_output_fits}")

CameraSettings loaded: /Volumes/SSD-PEU4A/Images/2026-04-05/EKRAN 12/19_45_10.CameraSettings.txt

FITS export completed.
Input SER       : /Volumes/SSD-PEU4A/Images/2026-04-05/EKRAN 12/19_45_10.ser
Frame index     : 945
Frame count     : 1275
Image size      : 5880 × 4422 pixels
Camera metadata : True
Output FITS     : /Users/kiyoaki/VScode/magzero/output/ser2fits/19_45_10_frame_000946.fits
